# Stage 2b — GRPO on Colab A100

Same GRPO training as Stage 2 but optimised for the A100 GPU on Colab Pro.
Larger batch size and relaxed hyperparameters compared to the Kaggle T4 version.

**Requires:** Stage 1 SFT adapter at `MyDrive/pokerapp/sft-adapter/`  
**Runtime:** A100 GPU (Colab Pro)  
**Output:** GRPO adapter saved to `MyDrive/pokerapp/grpo-adapter/`

### Before running
1. Runtime → Change runtime type → **A100 GPU**
2. Run with `MAX_STEPS = 50` first to get a timing estimate, then set to `2000` for the full run

## 1. Check GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU — change runtime to A100"
gpu = torch.cuda.get_device_properties(0)
print(f"GPU : {gpu.name}")
print(f"VRAM: {gpu.total_memory / 1e9:.1f} GB")
assert "A100" in gpu.name or gpu.total_memory > 30e9, "Not an A100 — switch runtime"

## 2. Install dependencies

In [ ]:
!pip install -q --upgrade unsloth
!pip install -q datasets "trl>=0.12.0" peft accelerate

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

SFT_ADAPTER_DIR  = "/content/drive/MyDrive/pokerapp/sft-adapter"
GRPO_ADAPTER_DIR = "/content/drive/MyDrive/pokerapp/grpo-adapter"
CHECKPOINT_DIR   = "/content/drive/MyDrive/pokerapp/grpo-checkpoints"

assert os.path.exists(SFT_ADAPTER_DIR), f"SFT adapter not found at {SFT_ADAPTER_DIR}"
os.makedirs(GRPO_ADAPTER_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"SFT adapter      : {SFT_ADAPTER_DIR} ✓")
print(f"GRPO adapter dir : {GRPO_ADAPTER_DIR}")
print(f"Checkpoint dir   : {CHECKPOINT_DIR}")

## 4. Clone repo

In [ ]:
import os, sys

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    repo_url = f"https://{token}@github.com/dominicvdb/pokerapp.git"
except Exception:
    repo_url = "https://github.com/dominicvdb/pokerapp.git"

if os.path.exists("/content/pokerapp"):
    !cd /content/pokerapp && git pull --quiet
    print("Repo updated")
else:
    !git clone {repo_url} /content/pokerapp --quiet
    print("Repo cloned")

sys.path.insert(0, "/content/pokerapp")

## 5. Load dataset

In [ ]:
from src.data_loader import load_pokerbench

dataset = load_pokerbench(cache_dir="/content/pokerapp/data")
train_ds = dataset["train"]

print(f"Train rows: {len(train_ds):,}")
print(f"Test rows : {len(dataset['test']):,}")

## 6. Load Qwen3-8B + SFT adapter

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_ADAPTER_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

print("SFT model loaded")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 7. Apply LoRA for GRPO

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing=False,
    random_state=42,
)

model.print_trainable_parameters()

## 8. Preprocess dataset

In [ ]:
from src.preprocessor import format_grpo, apply_chat_template

def preprocess_for_grpo(row):
    return {
        "prompt": apply_chat_template(
            format_grpo(row), tokenizer, add_generation_prompt=True
        ),
        "answer": row["output"],
    }

grpo_dataset = train_ds.map(
    preprocess_for_grpo,
    remove_columns=train_ds.column_names,
)

print(f"Processed {len(grpo_dataset):,} rows")

## 9. Define reward function

In [ ]:
from src.reward import poker_reward

def grpo_reward_fn(completions, answer=None, **kwargs):
    return [poker_reward(c, a) for c, a in zip(completions, answer)]

# Sanity check
print("Reward check:", grpo_reward_fn(["bet 18", "fold", "call"], answer=["bet 18", "fold", "raise 10"]))

## 10. Train with GRPOTrainer

**A100 vs T4 differences:**
- `per_device_train_batch_size=4` (was 2 on T4) — A100 has ~5x more VRAM
- `max_completion_length=64` (was 32 on T4)
- `bf16=True` always — A100 supports bf16 natively

**Hyperparameters (relaxed from T4 run):**
- `learning_rate=4e-5` — more room to learn than 2e-5
- `beta=0.05` — lighter KL penalty, lets reward signal dominate
- `max_grad_norm=0.3` — stable but not over-clipped

**Set `MAX_STEPS = 50` first** to get a timing estimate, then change to `2000` for the full run.

In [ ]:
from trl import GRPOTrainer, GRPOConfig
import time

# Set to 50 for a timing smoke test, 2000 for full training
MAX_STEPS = 50

config = GRPOConfig(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=1,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=4,   # A100: double the T4 batch size
    gradient_accumulation_steps=2,
    learning_rate=4e-5,
    beta=0.05,                       # lighter KL penalty than T4 run
    num_generations=4,
    max_completion_length=64,        # A100 can afford longer completions
    max_prompt_length=512,
    max_grad_norm=0.3,
    bf16=True,                       # A100 always supports bf16
    fp16=False,
    gradient_checkpointing=False,
    warmup_steps=min(100, MAX_STEPS // 5),
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=grpo_reward_fn,
    args=config,
    train_dataset=grpo_dataset,
)

print(f"Max steps       : {MAX_STEPS}")
print(f"Effective batch : {4 * 2} prompts × 4 generations = {4 * 2 * 4} completions per update")

In [ ]:
import os, time

checkpoints = [
    d for d in os.listdir(CHECKPOINT_DIR)
    if d.startswith("checkpoint-")
] if os.path.exists(CHECKPOINT_DIR) else []

resume_from = CHECKPOINT_DIR if checkpoints else None
if resume_from:
    latest = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
    print(f"Resuming from checkpoint: {latest}")
else:
    print("Starting from scratch")

t0 = time.time()
trainer_stats = trainer.train(resume_from_checkpoint=resume_from)
elapsed = time.time() - t0

steps_done = trainer_stats.metrics.get('train_steps_per_second', None)
it_per_sec = MAX_STEPS / elapsed
estimated_2000 = 2000 / it_per_sec / 3600

print(f"\nTraining complete")
print(f"Steps run    : {MAX_STEPS}")
print(f"Time elapsed : {elapsed / 60:.1f} min")
print(f"Speed        : {it_per_sec:.2f} it/s")
print(f"")
print(f"--- Timing estimate ---")
print(f"2000 steps   : ~{estimated_2000:.1f} hrs")
print(f"5000 steps   : ~{5000 / it_per_sec / 3600:.1f} hrs")

## 11. Full training run

Once happy with the timing estimate above, change `MAX_STEPS = 2000` in cell 10 and re-run cells 10 and 11.
The trainer will auto-resume from the last checkpoint if the session is interrupted.

## 12. Save GRPO adapter to Google Drive

In [ ]:
model.save_pretrained(GRPO_ADAPTER_DIR)
tokenizer.save_pretrained(GRPO_ADAPTER_DIR)

print(f"GRPO adapter saved to {GRPO_ADAPTER_DIR}")
!ls -lh {GRPO_ADAPTER_DIR}

## 13. Spot check — compare SFT vs GRPO

Runs 20 test examples. Compare against the 75% SFT baseline.

In [ ]:
import re
from src.reward import poker_reward

FastLanguageModel.for_inference(model)
_think_re = re.compile(r"<think>.*?</think>", re.DOTALL)

test_ds = dataset["test"].select(range(20))
correct = 0

for row in test_ds:
    prompt = apply_chat_template(
        format_grpo(row), tokenizer, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()
    generated_clean = _think_re.sub("", generated).strip()

    reward = poker_reward(generated_clean, row["output"])
    if reward == 1.0:
        correct += 1
    print(f"Expected: {row['output']:<12}  Predicted: {generated_clean:<12}  Reward: {reward}")

print(f"\nGRPO spot-check accuracy: {correct}/20 ({correct * 5}%)")
print(f"SFT baseline was 75% — improvement: {correct * 5 - 75:+}%")